# QDGrasp Phase 4 — CUDA gate for QDGrasp-Flow

Runs `ROADMAP-P4-001`'s CPU gate and then the CUDA gate on a fresh Kaggle or
Colab GPU runtime.

**Nothing here is a grasping result.** §7 of the plan forbids citing any P4
number as one. What this notebook produces is architecture evidence: does the
model run forward and backward on real NVIDIA hardware, does every parameter get
a gradient, do the outputs stay valid, does CUDA agree with CPU, and does memory
grow linearly rather than quadratically with the point count.

**The GPU cell may refuse to run, and that refusal is a result.** The harness
will not label a CPU run as CUDA (`ADR-0006`). If the runtime has no accelerator
the verdict is `refused` and P4-11 stays open — that is the harness working, not
a notebook bug to route around.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CODE_REVISION = "REPLACE_WITH_PUSHED_COMMIT"
REPO_URL = "https://github.com/ninicom/qdgrasp.git"
REPO_DIR = Path("/tmp/qdgrasp_repo")
WORK = Path("/kaggle/working/p4") if Path("/kaggle/working").is_dir() else Path("/content/p4")
WORK.mkdir(parents=True, exist_ok=True)

assert sys.version_info >= (3, 11), f"Python >=3.11 required, got {sys.version}"
assert CODE_REVISION != "REPLACE_WITH_PUSHED_COMMIT", (
    "Pin a commit that exists on origin. The point of pinning is that the evidence names "
    "the exact code that produced it; a branch name would let the code move underneath the result."
)

def run(*command, cwd=None, check=True):
    print("$", " ".join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], cwd=cwd, check=check)

if not REPO_DIR.is_dir():
    run("git", "clone", "--filter=blob:none", REPO_URL, REPO_DIR)
run("git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", CODE_REVISION)
run("git", "-C", REPO_DIR, "checkout", "--detach", CODE_REVISION)
head = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
assert head.stdout.strip() == CODE_REVISION, head.stdout
print("pinned at", CODE_REVISION)

In [ ]:
# P4 needs no simulator: the model layer is torch, pydantic and the robot
# profiles. MuJoCo is installed anyway because RobotSpec parses MJCF profiles.
run(sys.executable, "-m", "pip", "install", "--quiet",
    "mujoco>=3.3.0", "pydantic>=2.10.0", "PyYAML>=6.0.0", "scipy>=1.14.0", "trimesh>=4.0.0")
run(sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR))

ASSETS_DIR = Path("/tmp/robot-assets/mujoco-menagerie")
MENAGERIE_REVISION = "da76818e269b82289eba39808e2fb91d679d6994"
if not ASSETS_DIR.is_dir():
    ASSETS_DIR.parent.mkdir(parents=True, exist_ok=True)
    run("git", "clone", "--filter=blob:none",
        "https://github.com/google-deepmind/mujoco_menagerie.git", ASSETS_DIR)
run("git", "-C", ASSETS_DIR, "checkout", "--detach", MENAGERIE_REVISION)
os.environ["QDGRASP_ROBOT_ASSETS"] = str(ASSETS_DIR.parent)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no device")

## Stage 1 — the CPU gate

Ten of the thirteen packages should report `PASS`, and the checker should exit
non-zero on the three that are open. A zero exit here would mean the gate
stopped telling the truth about what is unfinished.

In [ ]:
cpu_gate = subprocess.run(
    [sys.executable, "scripts/check_phase4.py", "--profile", "micro",
     "--json", str(WORK / "phase4_cpu_gate.json")],
    check=False,
)
print("exit code:", cpu_gate.returncode, "(1 is expected while packages are open)")

## Stage 2 — the CPU overfit reference

The same diagnostic that ran on the development machine, so the CUDA numbers
have a reference from this runtime rather than only from a different one.

The verdict is read off pose error, not the total loss: the flow term has an
irreducible floor, because the velocity target is stochastic given the
interpolated state and the time.

In [ ]:
run(sys.executable, "scripts/overfit_qdgrasp_flow.py",
    "--device", "cpu", "--steps", "600",
    "--report", str(WORK / "phase4_overfit_cpu.json"), check=False)
print(open(WORK / "phase4_overfit_cpu.json").read()[:1500])

## Stage 3 — the CUDA gate

Forward, backward, gradient coverage, output validity, CPU/CUDA FP32 parity,
memory scaling and a short overfit — for **both** active hands (LEAP and Wonik
Allegro; Shadow stays paused under `ADR-0008`).

If this cell refuses, keep the output. `verdict: refused` is the evidence that
the gate was run and could not be met on this runtime.

In [ ]:
import torch
if not torch.cuda.is_available():
    print("No CUDA on this runtime. NOTHING here may be reported as CUDA evidence (ADR-0006).")
    print("Switch the runtime to a GPU accelerator and re-run from Stage 3.")
gpu = subprocess.run(
    [sys.executable, "scripts/phase4_cuda_gate.py",
     "--device", "cuda:0", "--points", "1024", "--steps", "200",
     "--evidence", str(WORK / "phase4_cuda_evidence.json")],
    check=False,
)
print("exit code:", gpu.returncode)
print(open(WORK / "phase4_cuda_evidence.json").read()[:4000])

## Stage 4 — what to carry back

Download `phase4_cpu_gate.json`, `phase4_overfit_cpu.json` and
`phase4_cuda_evidence.json`. Commit them under `evidence/phase4/` with the
commit this notebook pinned; the CUDA record belongs at
`evidence/phase4/cuda-<gpu>-<date>.json`, which is where `check_phase4.py`
looks for it.

P4-11 closes on a run whose verdict is `measured` **and** whose parity holds for
both hands. Any other verdict is recorded as-is and the package stays open.

In [ ]:
for name in ("phase4_cpu_gate.json", "phase4_overfit_cpu.json", "phase4_cuda_evidence.json"):
    path = WORK / name
    print(f"{name:32s} {'present' if path.is_file() else 'absent':8s} "
          f"{path.stat().st_size if path.is_file() else 0} bytes")